# Fase 5: exportar el generador ajustado a GGUF (LetraDeMédico)

Fusiona el adaptador LoRA de la Fase 4 con Qwen2.5-3B-Instruct y lo convierte a **GGUF Q4_K_M** (unos 2 GB), el formato de llama.cpp. Así el generador corre en la CPU de un computador normal, dentro de la app.

No necesita GPU: sirve el entorno de CPU de Colab (o la T4). Tarda unos 15 a 20 minutos en total. El archivo final se guarda en Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/letrademedico'
ADAPTADOR = f'{DRIVE}/qwen2.5-3b-instruct-lora'
SALIDA = f'{DRIVE}/letrademedico-qwen2.5-3b-q4_k_m.gguf'
!ls {ADAPTADOR}

## 1. Fusionar LoRA con el modelo base

Se carga el modelo base en float16 (no en 4 bits) y se suma BA a cada matriz W: el resultado es un modelo normal, sin adaptador, con el ajuste ya incorporado.

In [ ]:
# Colab trae torchao 0.10, que la versión actual de peft rechaza; no se usa, así que se quita
!pip uninstall -y -q torchao
!pip install -q -U transformers peft accelerate
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-3B-Instruct', dtype=torch.float16, device_map='cpu')
fusionado = PeftModel.from_pretrained(base, ADAPTADOR).merge_and_unload()
fusionado.save_pretrained('/content/fusionado', safe_serialization=True)
AutoTokenizer.from_pretrained(ADAPTADOR).save_pretrained('/content/fusionado')
del base, fusionado
!ls -lh /content/fusionado

## 2. Convertir a GGUF y cuantizar a 4 bits

In [ ]:
%cd /content
!test -d llama.cpp || git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q gguf sentencepiece protobuf
!python llama.cpp/convert_hf_to_gguf.py /content/fusionado --outfile /content/ldm-f16.gguf --outtype f16
!ls -lh /content/ldm-f16.gguf

In [ ]:
# Compila solo la herramienta de cuantización (unos minutos)
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build llama.cpp/build --target llama-quantize -j 4 > /dev/null
!llama.cpp/build/bin/llama-quantize /content/ldm-f16.gguf "{SALIDA}" Q4_K_M
!ls -lh "{SALIDA}"

## 3. Descargar

Descarga el `.gguf` desde Google Drive (`MyDrive/letrademedico/`) y cópialo en `models/generador/` del proyecto local. La descarga directa desde Colab de un archivo de 2 GB suele fallar, por eso conviene hacerlo desde Drive.